# New version, using the .parquet exports
instead of synchronizing the complete postgresql database all the time, i can export PARAM and METRIC .parquet files per group_id and only synchronize/copy what i need.

In [1]:
import sys
import os
project_root = os.path.abspath('..')          # one level up from notebooks directory
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
import numpy as np

In [2]:
tag = "2026-03-31_12-34_hpc3_P3.1.10"
PARAMS = pd.read_parquet(f"../data/experiments/exports/['{tag}']-params.parquet")
METRICS = pd.read_parquet(f"../data/experiments/exports/['{tag}']-metrics.parquet")

In [ ]:
from key_mapping import key_mapping
from data_preparation import add_grouped_mean_columns, get_multi_carrier_metrics

METRICS_WIDE = METRICS.pivot(columns='key', index=['run_uuid', 'step'], values='value').reset_index()
single_step_metrics = [col for col in METRICS_WIDE.columns if pd.isna(METRICS_WIDE[METRICS_WIDE['step'] > 0][col]).all()]
SS_METRICS = METRICS_WIDE[METRICS_WIDE['step'] == 0][['run_uuid'] + single_step_metrics]
SS_MERGED = pd.merge(PARAMS, SS_METRICS, on='run_uuid')
add_grouped_mean_columns(SS_MERGED, inplace=True)

# single step, multi-carrier metrics
SS_MC_METRICS = get_multi_carrier_metrics(SS_METRICS)
SS_MC_MERGED = pd.merge(PARAMS, SS_MC_METRICS, on='run_uuid')

# Multi-step metrics
multi_step_metrics = [col for col in METRICS_WIDE.columns if col not in single_step_metrics]
MS_METRICS = METRICS_WIDE[multi_step_metrics]
MS_METRICS.update(MS_METRICS.groupby('run_uuid').ffill())  # forward fill the multi-step metrics that might have gaps
MS_METRICS = MS_METRICS.replace(np.inf, np.nan)
add_grouped_mean_columns(MS_METRICS, inplace=True)
MS_MERGED = pd.merge(PARAMS, MS_METRICS, on='run_uuid')

# to wide format
SS_MC_MERGED_WIDE = SS_MC_MERGED.pivot(columns='key', values='value', index=PARAMS.columns.to_list() + ['carrier']).reset_index()
SS_MC_MERGED_WIDE = SS_MC_MERGED_WIDE.rename(columns=key_mapping)

updating value_mapping...
Identified groups: ['my_MinWBMP', 'my_convex_hull_jaccard_distance', 'my_hausdorff_distance', 'my_modified_hausdorff_distance', 'my_tsp_hull_jaccard_distance', 'my_tsp_obj_val_diff', 'target_opt_final']
Identified groups: ['target', 'target_opt']


In [49]:
non_plot_params = [
    'run_uuid',
    'name_x',
    'source_type',
    'source_name',
    'entry_point_name',
    'user_id',
    'status',
    'start_time',
    'end_time',
    'source_version',
    'lifecycle_stage_x',
    'artifact_uri',
    'experiment_id',
    'deleted_time',
    'name_y',
    'artifact_location',
    'lifecycle_stage_y',
    'creation_time',
    'last_update_time',
    'group_id',
    'instance_id',
    'mlflow.runName',
    'mlflow.source.git.commit',
    'mlflow.source.name',
    'mlflow.source.type',
    'mlflow.user',
    'mlflow.loggedArtifacts',
]
plot_params = [x for x in PARAMS.columns if x not in non_plot_params ]
non_unique_params: pd.DataFrame = PARAMS.loc[:, PARAMS.nunique() != 1]
non_unique_params = non_unique_params.drop(columns=non_plot_params, errors='ignore', inplace=False)
print("List of non-unique parameters:")
list(non_unique_params.columns)
# sort non-unique first
plot_params = [None] + list(non_unique_params) + [x for x in plot_params if x not in non_unique_params]

# get all metrics
plot_metrics = SS_MC_METRICS["key"].unique()

List of non-unique parameters:


## Plots

In [51]:
import ipywidgets as widgets
from ipywidgets import interact
import plotnine as p9
from my_theming import my_p9_theme, my_scale_color_and_fill

# 1. Add "None" to your params list for optional faceting
available_params = list(non_unique_params)
metrics = ["target_opt_final"]
manually_added = ["data__num_requests_per_carrier"]
parameter_selection = ["None", None] + available_params + metrics + manually_added

# 2. Define the plotting function
def update_plot(df, x, y, fill, linetype, cols, rows, hide_x_axis, scale_y_log10):
    ylab=df["Error Function"][0] if y == "target_opt_final" else y
    # Start building the plot
    p = (
        p9.ggplot(df,
                  p9.aes(x=x, y=y, fill=fill, linetype=linetype))
        + p9.geom_boxplot()
        + my_p9_theme
        + p9.theme(figure_size=(12, 6))
        + my_scale_color_and_fill
        + p9.labs(y=ylab)
    )
    if scale_y_log10:
        p += p9.scale_y_log10()
    if hide_x_axis:
        p += p9.theme(
        axis_text_x=p9.element_blank(),
        axis_ticks_x=p9.element_blank(),
        axis_title_x=p9.element_blank(),)
    
    # 3. Add faceting logic only if a parameter is selected
    if cols or rows:
        p += p9.facet_grid(rows=rows, cols=cols)

    group_keys = [key for key in [fill, cols, rows, linetype] if key not in [None, "None"]]
    n_per_series = df.groupby(group_keys)[y].count()
    print(f"n per boxplot: {n_per_series}")

    return p.draw() # Using .draw() ensures it renders correctly in the widget output

# 4. Create the interactive interface
interact(
    update_plot,
    df=widgets.fixed(SS_MC_MERGED_WIDE),
    x=widgets.Dropdown(options=plot_params, value=None, description='X Axis:'),
    y=widgets.Dropdown(options=plot_metrics, value="target_opt_final", description='Y Axis:'),
    fill=widgets.Dropdown(options=plot_params, value="Optimizer", description='Fill Color:'),
    linetype=widgets.Dropdown(options=plot_params, value="Optimizer", description='Linetype'),
    cols=widgets.Dropdown(options=plot_params, value="# clusters", description='Facet Cols:'),
    rows=widgets.Dropdown(options=plot_params, value="data__num_requests_per_carrier", description='Facet Rows:'),
    hide_x_axis=widgets.Checkbox(value=False, description="Hide x axis"),
    scale_y_log10=widgets.Checkbox(value=False, description="Log y axis"),
);

interactive(children=(Dropdown(description='X Axis:', options=(None, 'Optimizer', 'Optimizer Type', 'No. of Qu…